# Kuku promo finder: Colab runner

Makes **two kinds of promos** from a show's screenplays:
1. **Scene promos**: runs of **1, 2 and 3 consecutive episodes**, searched across **all episodes** (default: best 7 singles, 5 pairs, 3 triples = 15 promos)
2. **Theme montages**: the best moments of the **top 15 episodes**, joined across episodes by theme (default 6 promos)

Then every promo is checked, judged by a cold-viewer test plus a comparative judge, and ranked: `outputs/final_ranking.md`.

**Before you start**
1. In the Files panel (left), upload to `/content` a **zip of the show's episode `.txt` scripts** (for example `scripts.zip`).
   The code is downloaded from GitHub automatically (or upload `promo_ai.zip` yourself).
   Optional: `show_bible_fixed.json` (a hand-checked bible) and `outputs.zip` (results from an earlier session, to resume).
2. In 🔑 **Secrets** (left), add a free Gemini key (from aistudio.google.com) as **`GEMINI_API_KEY`** and switch on notebook access.
3. Run the cells **top to bottom**. Every stage saves its progress. If a cell stops on a quota error, run the **same cell** again later and it continues where it stopped.
4. Colab deletes files when the session ends, so **run the download cell** after long stages.

In [ ]:
# 1. Setup: get the code, unpack the scripts, restore earlier results if uploaded
REPO_URL = "https://github.com/deThorer/kuku-promo-finder"   # <- your repo

%cd /content
import os, glob, json, urllib.request
sh = get_ipython().system

if os.path.exists("/content/promo_ai.zip"):
    sh("unzip -o -q /content/promo_ai.zip -d /content")
elif not os.path.exists("/content/promo_ai/run_pipeline.py"):
    sh(f"git clone -q {REPO_URL} /content/promo_ai")
assert os.path.exists("/content/promo_ai/run_pipeline.py"), "Code not found: set REPO_URL or upload promo_ai.zip"

if not glob.glob("/content/promo_ai/input/*/*.txt") and not glob.glob("/content/promo_ai/input/*.txt"):
    zips = [z for z in glob.glob("/content/*.zip") if "promo_ai" not in z and "outputs" not in z]
    assert zips, "Upload a zip of the episode .txt scripts to /content first"
    sh(f'unzip -o -q "{zips[0]}" -d /content/promo_ai/input/')
if os.path.exists("/content/outputs.zip") and not os.path.exists("/content/promo_ai/outputs/moment_bank.json"):
    sh("unzip -o -q /content/outputs.zip -d /content/promo_ai/")
    print("Restored earlier results from outputs.zip")

%cd /content/promo_ai
print(len(glob.glob("input/**/*.txt", recursive=True)), "episode files found")

In [ ]:
# 2. API key + list the Flash models your key can use
from google.colab import userdata
key = None
for name in ("GEMINI_API_KEY", "GEMINI_API_KEY_2"):
    try:
        key = userdata.get(name)
        break
    except Exception:
        pass
assert key, "Add your key in Secrets as GEMINI_API_KEY (and allow notebook access)"
os.environ["GEMINI_API_KEY"] = key
os.environ["LLM_PROVIDER"] = "gemini"

req = urllib.request.Request("https://generativelanguage.googleapis.com/v1beta/models?pageSize=200",
                             headers={"x-goog-api-key": key})
models = json.load(urllib.request.urlopen(req))["models"]
print("\n".join(m["name"].removeprefix("models/") for m in models
                if "generateContent" in m.get("supportedGenerationMethods", []) and "flash" in m["name"]))

In [ ]:
# 3. Choose models. Each model has its OWN quota: check https://ai.dev/rate-limit
#    Flash models (free): 5 requests/minute, 20/day.  Flash-Lite models: 15/minute, 500/day.
BEST = "gemini-3.8-flash"        # 20/day: saved for the montages (hardest task: joining episodes into one story)
BULK = "gemini-3.5-flash-lite"   # 500/day: the 15 scene promos
LITE = "gemini-3.1-flash-lite"   # 500/day: moment bank (already built - skipped on reruns)

# If a model's DAILY quota runs out mid-stage, the tool switches to the next one automatically
os.environ["LLM_FALLBACK_MODELS"] = "gemini-3.5-flash-lite,gemini-3.1-flash-lite"
os.environ["LLM_MODEL"] = BULK
print("scenes ->", BULK, "| montages ->", BEST, "| fallback ->", os.environ["LLM_FALLBACK_MODELS"])

In [ ]:
# 4. Self-check (no AI, no quota): tests + a full dry run with simple rules
!python tests/test_offline.py | tail -1
!python run_pipeline.py all --provider baseline | tail -1

## Stage 1: Show bible (1 request)
Characters (with every speaker label), themes, and spoilers. **Check it**: the AI mixed up characters last time.
If you uploaded `show_bible_fixed.json`, that is used instead and no request is made.

In [ ]:
if os.path.exists("/content/show_bible_fixed.json"):
    os.makedirs("outputs", exist_ok=True)
    sh("cp /content/show_bible_fixed.json outputs/show_bible.json")
    print("Using your corrected bible: show_bible_fixed.json")
else:
    sh(f"python run_pipeline.py bible --model {MAIN}")
b = json.load(open("outputs/show_bible.json"))
print("\nCharacters:", [c["name"] for c in b["characters"]])
print("Themes:", [t["id"] for t in b["themes"]])
print("Spoilers:", [(r["episode"], r["reveal"][:60]) for r in b["big_reveals"]])

## Stage 2: Moment bank (~17 requests on LITE)
The 5 strongest moments of every episode, 3 episodes per request. Episodes that come back with fewer than 3 moments get one follow-up request. Episodes already in the bank are skipped.

In [ ]:
os.environ["LLM_MIN_INTERVAL"] = "5"     # Lite: 15 requests/minute -> one every 5s
!python run_pipeline.py moments --batch 3 --redo-thin --model {LITE}

## Stage 3: Rankings (0 requests)
Top 15 episodes, plus all 147 runs of 1-3 consecutive episodes scored. Picks: 7 singles, 5 pairs, 3 triples.

Rules applied in code:
- **Final act** (episode 36+, the last 30%) is penalised, and big-reveal episodes count as spoilers, because promos must not give away the ending.
- **Each episode is in at most one scene promo**, so the same best scene isn't cut twice.
- **Triples are picked first**, and windows scoring below 4.0 are skipped.

In [ ]:
!python run_pipeline.py rank --top 15 --per-run 7,5,3
from IPython.display import Markdown, display
display(Markdown(open("outputs/rankings.md").read()))

## Stage 4: Scene promos (15 promos, ~15-45 requests on 3.5 Flash Lite)
Runs of 1, 2 and 3 consecutive episodes across all 50. Finished promos are skipped on reruns, and promos from an older ranking are moved to `outputs/promos/_old/`.

In [ ]:
os.environ["LLM_MIN_INTERVAL"] = "5"     # Lite: 15 requests/minute
!python run_pipeline.py scenes --per-run 7,5,3 --model {BULK}

## Stage 5: Theme montages from the top 15 episodes (6 promos, ~7-13 requests on 3.8 Flash)
1 planning request (saved and reused on reruns), then one request per montage (+ repairs).
3.8 Flash allows 5 requests/minute, so the tool waits 13s between calls. If its 20/day run out, it falls back to the Lite models automatically.

In [ ]:
os.environ["LLM_MIN_INTERVAL"] = "13"    # Flash: 5 requests/minute -> one every 13s
!python run_pipeline.py montages --count 6 --top 15 --model {BEST}

## Stage 6: Recheck (0 requests)
Re-applies the checker to every promo that failed, including last-resort code fixes:
- a too-short segment is extended with the next whole lines
- a promo up to 15s too long gets a chunk cut from its **middle** at line boundaries (the hook and cliffhanger are kept)

Every code fix is written in the promo's notes.

In [ ]:
!python run_pipeline.py recheck

## Stage 7: Judge and final ranking (~4 requests on BEST)
1. **Cold-viewer test**: an AI with **no show context** reads each promo and says who and what it understood, what it wants to know, and what confused it.
2. **Judge**: compares **all promos in one call** (hook, cliffhanger, clarity, spoiler risk, duplicates).
3. **Code**: final score, with penalties for spoilers, failed rules and duplicate footage.

Writes `outputs/final_ranking.md`: the ranked list, the top 5 to post first with reasons, and a "where the tool picked badly" section.

In [ ]:
# Judge ALL promos together with ONE model (--redo), so every score is comparable. ~4 requests.
# BEST (3.8 Flash) resets at 1:30 PM IST; before that, use {BULK} instead of {BEST}.
os.environ["LLM_MIN_INTERVAL"] = "13"    # Flash: 5 requests/minute
!python run_pipeline.py judge --redo --model {BEST}
from IPython.display import Markdown, display
display(Markdown(open("outputs/final_ranking.md").read()))

## Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/promo_index.md").read()))   # every promo; the ranked list is in final_ranking.md

In [ ]:
# Download everything (raw outputs folder for the submission). Run after each long stage.
!zip -qr /content/outputs.zip outputs
from google.colab import files
files.download("/content/outputs.zip")

In [ ]:
# Optional: also keep a copy in your Google Drive (survives Colab resets)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    sh("mkdir -p /content/drive/MyDrive/kuku_promos && cp -r outputs /content/drive/MyDrive/kuku_promos/")
    print("Saved to MyDrive/kuku_promos/outputs")
except Exception as e:
    print("Drive not available (", e, ")")
    print("Skip this cell and use the download cell above instead. Drive needs you to approve the sign-in pop-up;")
    print("if it fails, your browser may be blocking third-party cookies for colab.research.google.com.")